# MLP Final Selected Model

This notebook loads `selected_hyperparameters.json`, retrains exactly that confirmed configuration, and is the only MLP notebook that may evaluate the frozen test set. Do not change hyperparameters after viewing test results.


## 1. Package Setup


In [ ]:
import hashlib
import json
import os
import random
import time
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 7
EARLY_STOP_MIN_DELTA = 1e-4
THRESHOLD = 0.5
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)


def resolve_project_root():
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


PROJECT_ROOT = resolve_project_root()
SHARED_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "shared"
SHARED_MANIFESTS_DIR = SHARED_OUTPUT_DIR / "manifests"
SHARED_CLASS_WEIGHT_PATH = SHARED_OUTPUT_DIR / "class_weights.json"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "mlp"
TABLES_DIR = OUTPUT_DIR / "tables"
CACHE_DIR = OUTPUT_DIR / "cache"
HPO_DIR = OUTPUT_DIR / "hpo"
MODELS_DIR = OUTPUT_DIR / "models"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
for directory in [OUTPUT_DIR, TABLES_DIR, CACHE_DIR, HPO_DIR, MODELS_DIR, METRICS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}. Run Model Variants/Analysis/"
            "MLAAD_Scan_MFCC_Analysis_cache.ipynb, then 00_MLP_Data_Preparation.ipynb."
        )
    return path


def load_shared_class_weights(path):
    with open(require_file(path), "r", encoding="utf-8") as f:
        payload = json.load(f)
    weights = payload.get("class_weights", payload)
    return {int(label): float(weight) for label, weight in weights.items()}


required_cache_files = [
    CACHE_DIR / "X_train.npy",
    CACHE_DIR / "y_train.npy",
    CACHE_DIR / "train_metadata.csv",
    CACHE_DIR / "X_validation.npy",
    CACHE_DIR / "y_validation.npy",
    CACHE_DIR / "validation_metadata.csv",
    CACHE_DIR / "feature_config.json",
    SHARED_CLASS_WEIGHT_PATH,
]
for required in required_cache_files:
    require_file(required)

X_train = np.load(CACHE_DIR / "X_train.npy")
y_train = np.load(CACHE_DIR / "y_train.npy")
train_metadata = pd.read_csv(CACHE_DIR / "train_metadata.csv")
X_validation = np.load(CACHE_DIR / "X_validation.npy")
y_validation = np.load(CACHE_DIR / "y_validation.npy")
validation_metadata = pd.read_csv(CACHE_DIR / "validation_metadata.csv")

# Keep these names for older checklist cells, but they now point to cache metadata copied
# from the single canonical shared manifest rather than a model-local split.
train_manifest = train_metadata.copy()
validation_manifest = validation_metadata.copy()

with open(CACHE_DIR / "feature_config.json", "r", encoding="utf-8") as f:
    feature_config = json.load(f)
cnn_feature_config = feature_config

CLASS_WEIGHTS = load_shared_class_weights(SHARED_CLASS_WEIGHT_PATH)
class_weights = CLASS_WEIGHTS

if len(X_train) != len(y_train) or len(X_train) != len(train_metadata):
    raise RuntimeError("Training arrays, labels and metadata do not have the same row count.")
if len(X_validation) != len(y_validation) or len(X_validation) != len(validation_metadata):
    raise RuntimeError("Validation arrays, labels and metadata do not have the same row count.")

print("Loaded MLP-ready cache:", CACHE_DIR)
print("Input representation:", "80-D MFCC mean/std vector")
print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("Shared class weights:", CLASS_WEIGHTS)


## 2. Load Shared MLP-Ready Cache


In [ ]:
import hashlib
import json
import os
import random
import time
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 7
EARLY_STOP_MIN_DELTA = 1e-4
THRESHOLD = 0.5
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)


def resolve_project_root():
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


PROJECT_ROOT = resolve_project_root()
SHARED_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "shared"
SHARED_MANIFESTS_DIR = SHARED_OUTPUT_DIR / "manifests"
SHARED_CLASS_WEIGHT_PATH = SHARED_OUTPUT_DIR / "class_weights.json"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "mlp"
TABLES_DIR = OUTPUT_DIR / "tables"
CACHE_DIR = OUTPUT_DIR / "cache"
HPO_DIR = OUTPUT_DIR / "hpo"
MODELS_DIR = OUTPUT_DIR / "models"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
for directory in [OUTPUT_DIR, TABLES_DIR, CACHE_DIR, HPO_DIR, MODELS_DIR, METRICS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}. Run Model Variants/Analysis/MLAAD_Scan_MFCC_Analysis_cache.ipynb, "
            "then 00_MLP_Data_Preparation.ipynb."
        )
    return path


def load_shared_class_weights(path):
    with open(require_file(path), "r", encoding="utf-8") as f:
        payload = json.load(f)
    weights = payload.get("class_weights", payload)
    return {int(label): float(weight) for label, weight in weights.items()}


for required in [
    CACHE_DIR / "X_train.npy",
    CACHE_DIR / "y_train.npy",
    CACHE_DIR / "train_metadata.csv",
    CACHE_DIR / "X_validation.npy",
    CACHE_DIR / "y_validation.npy",
    CACHE_DIR / "validation_metadata.csv",
    CACHE_DIR / "feature_config.json",
    SHARED_CLASS_WEIGHT_PATH,
]:
    require_file(required)

X_train = np.load(CACHE_DIR / "X_train.npy")
y_train = np.load(CACHE_DIR / "y_train.npy")
train_metadata = pd.read_csv(CACHE_DIR / "train_metadata.csv")
X_validation = np.load(CACHE_DIR / "X_validation.npy")
y_validation = np.load(CACHE_DIR / "y_validation.npy")
validation_metadata = pd.read_csv(CACHE_DIR / "validation_metadata.csv")

# Compatibility aliases for older checklist cells. These are cache metadata copied from the
# single canonical shared manifest, not a model-local split.
train_manifest = train_metadata.copy()
validation_manifest = validation_metadata.copy()

with open(CACHE_DIR / "feature_config.json", "r", encoding="utf-8") as f:
    feature_config = json.load(f)
cnn_feature_config = feature_config

CLASS_WEIGHTS = load_shared_class_weights(SHARED_CLASS_WEIGHT_PATH)
class_weights = CLASS_WEIGHTS

if len(X_train) != len(y_train) or len(X_train) != len(train_metadata):
    raise RuntimeError("Training arrays, labels and metadata do not have the same row count.")
if len(X_validation) != len(y_validation) or len(X_validation) != len(validation_metadata):
    raise RuntimeError("Validation arrays, labels and metadata do not have the same row count.")

print("Loaded MLP-ready cache:", CACHE_DIR)
print("Input representation:", "80-D MFCC mean/std vector")
print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("Shared class weights:", CLASS_WEIGHTS)


## 3. Shared MLP Model Builder


In [ ]:
SEARCH_SPACE = {
    "hidden_units": [
        [64],
        [128],
        [64, 32],
        [128, 64],
        [256, 128],
        [256, 128, 64],
    ],
    "activation": ["relu", "tanh"],
    "dropout": [0.0, 0.1, 0.2, 0.3, 0.4],
    "learning_rate": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
    "batch_size": [32, 64, 128, 256],
}
SEARCH_KEYS = ["hidden_units", "activation", "dropout", "learning_rate", "batch_size"]
EXPECTED_SEARCH_SPACE_SIZE = 1200


def normalise_config(config):
    return {
        "hidden_units": [int(value) for value in config["hidden_units"]],
        "activation": str(config["activation"]),
        "dropout": float(config["dropout"]),
        "learning_rate": float(config["learning_rate"]),
        "batch_size": int(config["batch_size"]),
    }


def enumerate_search_space():
    configs = []
    for values in product(*(SEARCH_SPACE[key] for key in SEARCH_KEYS)):
        configs.append(normalise_config(dict(zip(SEARCH_KEYS, values))))
    return configs


ALL_CONFIGURATIONS = enumerate_search_space()
if len(ALL_CONFIGURATIONS) != EXPECTED_SEARCH_SPACE_SIZE:
    raise RuntimeError(f"Expected 1200 configurations, found {len(ALL_CONFIGURATIONS)}")


def config_json(config):
    return json.dumps(normalise_config(config), sort_keys=True)


def config_key(config):
    return hashlib.sha256(config_json(config).encode("utf-8")).hexdigest()


def seed_from_config(config, base_seed=RANDOM_STATE):
    digest = hashlib.sha256(f"{base_seed}:{config_json(config)}".encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % (2**31 - 1)


def set_global_seed(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def build_mlp_model(config, input_dim):
    config = normalise_config(config)
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    for units in config["hidden_units"]:
        model.add(Dense(units, activation=config["activation"]))
        model.add(Dropout(config["dropout"]))
    model.add(Dense(1, activation="sigmoid"))
    optimizer = tf.keras.optimizers.Adam(learning_rate=config["learning_rate"])
    model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=["accuracy"])
    return model


def flatten_config(config):
    config = normalise_config(config)
    return {
        "hidden_units": json.dumps(config["hidden_units"]),
        "activation": config["activation"],
        "dropout": config["dropout"],
        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "config_json": config_json(config),
        "config_key": config_key(config),
    }


def train_and_evaluate_config(config, run_seed, run_name, verbose=0):
    config = normalise_config(config)
    tf.keras.backend.clear_session()
    set_global_seed(run_seed)
    model = build_mlp_model(config, X_train.shape[1])

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOP_PATIENCE,
            min_delta=EARLY_STOP_MIN_DELTA,
            restore_best_weights=True,
        )
    ]
    start_time = time.perf_counter()
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=MAX_EPOCHS,
        batch_size=config["batch_size"],
        class_weight=CLASS_WEIGHTS,
        callbacks=callbacks,
        verbose=verbose,
    )
    runtime_seconds = time.perf_counter() - start_time

    validation_probability = model.predict(X_validation, batch_size=config["batch_size"], verbose=0).ravel()
    validation_pred = (validation_probability >= THRESHOLD).astype(int)
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    best_val_loss = float(np.min(history.history["val_loss"]))

    return {
        "run_name": run_name,
        "seed": int(run_seed),
        "validation_macro_f1": f1_score(y_validation, validation_pred, average="macro", zero_division=0),
        "validation_binary_f1_synthetic": f1_score(y_validation, validation_pred, pos_label=1, zero_division=0),
        "validation_accuracy": accuracy_score(y_validation, validation_pred),
        "validation_precision_synthetic": precision_score(y_validation, validation_pred, pos_label=1, zero_division=0),
        "validation_recall_synthetic": recall_score(y_validation, validation_pred, pos_label=1, zero_division=0),
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "epochs_trained": int(len(history.history["loss"])),
        "early_stopped": bool(len(history.history["loss"]) < MAX_EPOCHS),
        "runtime_seconds": float(runtime_seconds),
        "objective": "validation_macro_f1",
        "threshold": THRESHOLD,
        **flatten_config(config),
    }


print("Shared HPO search-space size:", len(ALL_CONFIGURATIONS))


## 4. Final Selected Model Test Evaluation


In [ ]:
RUN_FINAL_TEST_EVALUATION = False
VERBOSE_TRAINING = 2

selected_hyperparameters_path = MODELS_DIR / "selected_hyperparameters.json"
final_model_path = MODELS_DIR / "mlp_final_selected_model.keras"
final_validation_metrics_path = METRICS_DIR / "mlp_final_validation_metrics.json"
test_metrics_path = METRICS_DIR / "mlp_test_metrics.json"
confusion_matrix_path = METRICS_DIR / "mlp_confusion_matrix.csv"
language_subgroup_path = METRICS_DIR / "mlp_language_subgroup_metrics.csv"
tts_generator_subgroup_path = METRICS_DIR / "mlp_tts_generator_subgroup_metrics.csv"
confusion_plot_path = FIGURES_DIR / "mlp_confusion_matrix.png"


def subgroup_metrics(metadata, y_true, y_pred, group_column, min_rows=5):
    if group_column not in metadata.columns:
        return pd.DataFrame()
    working = metadata.copy().reset_index(drop=True)
    working["y_true"] = y_true
    working["y_pred"] = y_pred
    rows = []
    for group_value, group in working.groupby(group_column):
        if len(group) < min_rows:
            continue
        rows.append(
            {
                group_column: group_value,
                "files": int(len(group)),
                "macro_f1": f1_score(group["y_true"], group["y_pred"], average="macro", zero_division=0),
                "binary_f1_synthetic": f1_score(group["y_true"], group["y_pred"], pos_label=1, zero_division=0),
                "accuracy": accuracy_score(group["y_true"], group["y_pred"]),
                "precision_synthetic": precision_score(group["y_true"], group["y_pred"], pos_label=1, zero_division=0),
                "recall_synthetic": recall_score(group["y_true"], group["y_pred"], pos_label=1, zero_division=0),
            }
        )
    return pd.DataFrame(rows)


if RUN_FINAL_TEST_EVALUATION:
    require_file(selected_hyperparameters_path)
    require_file(TABLES_DIR / "test_manifest.csv")
    require_file(CACHE_DIR / "X_test.npy")
    require_file(CACHE_DIR / "y_test.npy")
    require_file(CACHE_DIR / "test_metadata.csv")

    with open(selected_hyperparameters_path, "r", encoding="utf-8") as f:
        selected_hyperparameters = json.load(f)
    final_config = selected_hyperparameters["configuration"]

    final_validation_row = train_and_evaluate_config(
        final_config,
        run_seed=RANDOM_STATE,
        run_name="final_selected_model_validation_fit",
        verbose=VERBOSE_TRAINING,
    )
    with open(final_validation_metrics_path, "w", encoding="utf-8") as f:
        json.dump(final_validation_row, f, indent=2)

    tf.keras.backend.clear_session()
    set_global_seed(RANDOM_STATE)
    final_model = build_mlp_model(final_config, X_train.shape[1])
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOP_PATIENCE,
            min_delta=EARLY_STOP_MIN_DELTA,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ModelCheckpoint(str(final_model_path), monitor="val_loss", save_best_only=True),
    ]
    final_model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=MAX_EPOCHS,
        batch_size=final_config["batch_size"],
        class_weight=CLASS_WEIGHTS,
        callbacks=callbacks,
        verbose=VERBOSE_TRAINING,
    )

    X_test = np.load(CACHE_DIR / "X_test.npy")
    y_test = np.load(CACHE_DIR / "y_test.npy")
    test_metadata = pd.read_csv(CACHE_DIR / "test_metadata.csv")
    test_probability = final_model.predict(X_test, batch_size=final_config["batch_size"], verbose=0).ravel()
    test_pred = (test_probability >= THRESHOLD).astype(int)

    test_metrics = {
        "model": "MLP",
        "input_representation": "40 MFCC mean/std aggregated features",
        "selected_configuration": final_config,
        "source_hpo_method": selected_hyperparameters.get("source_hpo_method"),
        "test_macro_f1": f1_score(y_test, test_pred, average="macro", zero_division=0),
        "test_binary_f1_synthetic": f1_score(y_test, test_pred, pos_label=1, zero_division=0),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "test_precision_synthetic": precision_score(y_test, test_pred, pos_label=1, zero_division=0),
        "test_recall_synthetic": recall_score(y_test, test_pred, pos_label=1, zero_division=0),
        "threshold": THRESHOLD,
    }
    with open(test_metrics_path, "w", encoding="utf-8") as f:
        json.dump(test_metrics, f, indent=2)

    cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
    pd.DataFrame(cm, index=[CLASS_NAMES[0], CLASS_NAMES[1]], columns=[CLASS_NAMES[0], CLASS_NAMES[1]]).to_csv(confusion_matrix_path)

    language_metrics_df = subgroup_metrics(test_metadata, y_test, test_pred, "language")
    tts_generator_metrics_df = subgroup_metrics(test_metadata, y_test, test_pred, "tts_generator")
    if not language_metrics_df.empty:
        language_metrics_df.to_csv(language_subgroup_path, index=False)
    if not tts_generator_metrics_df.empty:
        tts_generator_metrics_df.to_csv(tts_generator_subgroup_path, index=False)

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[CLASS_NAMES[0], CLASS_NAMES[1]], yticklabels=[CLASS_NAMES[0], CLASS_NAMES[1]])
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.title("MLP final test confusion matrix")
    plt.tight_layout()
    plt.savefig(confusion_plot_path, dpi=300, bbox_inches="tight")
    plt.show()

    display(pd.DataFrame([test_metrics]))
    if not language_metrics_df.empty:
        display(language_metrics_df.sort_values("macro_f1", ascending=False))
    if not tts_generator_metrics_df.empty:
        display(tts_generator_metrics_df.sort_values("macro_f1", ascending=False))
else:
    print("RUN_FINAL_TEST_EVALUATION is False. Set it to True only after confirmation selects final hyperparameters.")


## 5. Final Reproducibility Checklist


In [ ]:
checks = {
    "selected_hyperparameters_loaded": "only when RUN_FINAL_TEST_EVALUATION is True",
    "final_test_evaluation_allowed_here": True,
    "subgroup_columns": ["language", "tts_generator"],
    "test_metrics_path": str(METRICS_DIR / "mlp_test_metrics.json"),
    "do_not_retune_after_test": True,
}
display(pd.DataFrame(list(checks.items()), columns=["check", "value"]))
print("Final result placeholders remain empty until RUN_FINAL_TEST_EVALUATION is intentionally set to True and this notebook is run.")
